# memrust — quickstart

**Memory infrastructure for AI agents.** memrust is an agent-native memory
engine written in Rust: hybrid retrieval (vector + BM25 + entity graph +
recency) behind a `remember` / `recall` / `forget` API, where every result
explains *why* it surfaced with a per-signal score breakdown.

This notebook gets you from zero to recall in about two minutes:

1. install the Python SDK and start the engine
2. remember a few memories
3. recall them — and read the signal breakdown
4. see the strategies (`lexical` for exact IDs, `relational` for the entity graph)
5. open the built-in web dashboard from Colab

Repo: https://github.com/AIAnytime/memrust · PyPI: `pip install memrust`

In [ ]:
%pip install -q memrust

In [ ]:
# Download the memrust server for this machine and start it. One static
# binary, no Rust toolchain needed. Works in Colab (Linux x86_64) and on a
# local Mac; for any other platform, build with `cargo build --release`
# and set BIN to target/release/memrust.
import os, platform, subprocess, time, urllib.request

VERSION = "v0.6.1"
DATA_DIR = "./memory-quickstart"          # this notebook's own memory store

def release_target():
    system, machine = platform.system().lower(), platform.machine().lower()
    if system == "darwin":
        return "aarch64-apple-darwin" if machine in ("arm64", "aarch64") else "x86_64-apple-darwin"
    if system == "linux" and machine in ("x86_64", "amd64"):
        return "x86_64-unknown-linux-musl"
    raise RuntimeError(
        f"no prebuilt binary for {system}/{machine} — "
        "run `cargo build --release` and point BIN at target/release/memrust"
    )

TARGET = release_target()
# Keyed by target so a binary for the wrong platform is never reused.
BIN = f"./memrust-bin/{TARGET}/memrust"

if not os.path.exists(BIN):
    os.makedirs(os.path.dirname(BIN), exist_ok=True)
    url = (f"https://github.com/AIAnytime/memrust/releases/download/"
           f"{VERSION}/memrust-{VERSION}-{TARGET}.tar.gz")
    urllib.request.urlretrieve(url, "memrust-bin/memrust.tar.gz")
    subprocess.run(["tar", "xzf", "../memrust.tar.gz"],
                   cwd=os.path.dirname(BIN), check=True)
    os.chmod(BIN, 0o755)

# Re-running this cell (or another memrust notebook in the same runtime) can
# leave an old server holding port 7700 with a different data dir — replace it.
subprocess.run(["pkill", "-f", "memrust serve"], capture_output=True)
time.sleep(0.5)

server = subprocess.Popen(
    [BIN, "serve", "--data-dir", DATA_DIR,
     "--lifecycle-interval-secs", "0",     # we trigger lifecycle explicitly below
     "--consolidate-after-secs", "0"],     # so consolidation demos run immediately
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:7700/health", timeout=1)
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("memrust did not come up — check that port 7700 is free")
print(f"memrust {VERSION} ({TARGET}) is up on http://127.0.0.1:7700 "
      f"(data dir: memory-quickstart)")

In [ ]:
from memrust import MemrustClient

memory = MemrustClient("http://127.0.0.1:7700")
memory.health()

## Remember

Memories are typed the way agents think — `episodic` (things that happened),
`semantic` (distilled facts), `working` (short-lived scratch state, auto-expires),
`reflection`, `tool_call`, and `procedural` (how-to knowledge). Entities are
extracted automatically at ingest and feed the graph index.

In [ ]:
records = [
    memory.remember("Deploy 47 failed on cluster prod-west with error E1234; "
                    "root cause was an expired TLS cert",
                    kind="episodic", tags=["incident"], importance=0.9),
    memory.remember("To rotate TLS certs: run make certs, then restart the proxy",
                    kind="procedural", tags=["runbook"], importance=0.8),
    memory.remember("Project Phoenix depends on the Billing Service for invoicing",
                    kind="semantic"),
    memory.remember("the Billing Service rate limits at 100 requests per second",
                    kind="semantic"),
    memory.remember("Dana Whitfield leads Project Phoenix", kind="semantic"),
    memory.remember("User prefers concise answers with code examples", kind="semantic"),
]
for r in records:
    print(f'{r["kind"]:>10}  entities={r.get("entities", [])}')

## Recall — with explained scores

Every hit carries `signals`: the contribution from the **vector** index
(semantic similarity), **lexical** BM25 (exact terms), the entity **graph**
(related things), and **recency** decay. No opaque scores.

In [ ]:
def show(hits):
    """Pretty-print recall hits with the per-signal score breakdown."""
    for h in hits:
        s, r = h["signals"], h["record"]
        print(f'{h["score"]:.4f}  [{r["kind"]:>10}]  {r["text"][:88]}')
        print(f'          vector={s["vector"]:.4f}  lexical={s["lexical"]:.4f}  '
              f'graph={s["graph"]:.4f}  recency={s["recency"]:.2f}')

show(memory.recall("what went wrong with the deployment?", top_k=3))

## Strategies

Strategies reweight the fusion — they don't switch indexes off.

- `lexical`: exact identifiers that pure vector search whiffs on
- `relational`: follow entity links — *connected to X*, not just *similar to X*

In [ ]:
print("=== lexical: exact error code ===")
show(memory.recall("E1234", strategy="lexical", top_k=1))

print()
print("=== relational: the entity graph at work ===")
# 'Billing Service rate limits' never mentions Phoenix — it surfaces through
# a 1-hop walk in the entity graph (Phoenix <-> Billing Service). Watch its
# graph signal vs its lexical signal.
show(memory.recall("tell me about Project Phoenix", strategy="relational", top_k=3))

## Forget — durably

In [ ]:
victim = records[-1]["id"]
print("forgotten:", memory.forget(victim))
print("memories now:", memory.health()["total_memories"])

## The web dashboard

`memrust serve` embeds a full management dashboard (stat tiles, recall with
signal visualizations, memory browser, entity explorer, lifecycle controls).
In Colab, proxy the port:

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_window(7700)   # opens the memrust dashboard
except ImportError:
    print("Not running in Colab — open http://127.0.0.1:7700/ in a browser")

## Next

- **02_rag_with_embeddings.ipynb** — real embedding models, vector storage,
  hybrid retrieval, and a full RAG loop
- **03_pdf_rag_agents.ipynb** — PDF ingestion with pypdf, a LangGraph RAG
  agent, and the full memory feature set (lifecycle, multi-agent, snapshots)